[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/HyperCoast/blob/main/docs/examples/appeears_emit_point.ipynb)

# Request an EMIT point spectrum with AppEEARS

This notebook requests selected EMIT surface reflectance wavelengths for one point with the HyperCoast AppEEARS helpers.

In [ ]:
# %pip install "hypercoast[extra]"

In [ ]:
from pathlib import Path

import pandas as pd
import hypercoast

Log in and select EMIT reflectance layers by wavelength.

In [ ]:
client = hypercoast.appeears_login()

wavelengths = [490, 560, 650, 705, 865, 1640, 2210]
metadata = hypercoast.appeears_emit_layers(
    wavelengths,
    client=client,
    return_metadata=True,
)
layers = [{"product": item["product"], "layer": item["layer"]} for item in metadata]
layers

Build and submit a point task. Point coordinates use `(longitude, latitude)`.

In [ ]:
task = hypercoast.appeears_point_task(
    task_name="emit_hyperspectral_point",
    coordinates=(-118.33, 33.96),
    layers=layers,
    start_date="2024-04-01",
    end_date="2024-04-30",
)

response = hypercoast.appeears_submit_task(task, client=client)
response

Wait for the task and download the CSV result.

In [ ]:
task_id = response["task_id"]
hypercoast.appeears_wait(task_id, client=client, interval=60)

files = hypercoast.appeears_download(
    task_id,
    out_dir="data/appeears_emit_point",
    client=client,
    file_types=["csv"],
)
files

Read the point table and keep the EMIT band columns.

In [ ]:
csv_file = next(Path("data/appeears_emit_point").glob("*.csv"))
df = pd.read_csv(csv_file)
df.head()

In [ ]:
wavelength_by_layer = {item["layer"]: item["wavelength"] for item in metadata}
band_columns = [column for column in df.columns if "EMIT_L2A_RFL_001_B" in column]
wavelength_by_column = {
    column: wavelength_by_layer[column.split("_")[-1]] for column in band_columns
}
spectrum = df[band_columns].mean().rename(index=wavelength_by_column)
spectrum.index.name = "wavelength"
spectrum.plot(marker="o", xlabel="Wavelength (nm)", ylabel="Reflectance")